### 실습

In [1]:
import pandas as pd

In [2]:
from preamble import *

In [3]:
df = pd.read_csv("data/housing.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [4]:
df.isna().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [5]:
df["ocean_proximity"].value_counts() # 

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [6]:
df["ocean_proximity"].unique()

array(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'],
      dtype=object)

In [7]:
df = df.dropna() # 207개 삭제 > 왜? 첫 분석이므로 207/20640 처음에는 크게 크게 보기 위해서
df.head()

,longitude,latitude,housing_median_age,total_rooms,...,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,...,126.0,8.33,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,...,1138.0,8.30,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,...,177.0,7.26,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,...,219.0,5.64,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,...,259.0,3.85,342200.0,NEAR BAY


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20433 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20433 non-null  float64
 1   latitude            20433 non-null  float64
 2   housing_median_age  20433 non-null  float64
 3   total_rooms         20433 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20433 non-null  float64
 6   households          20433 non-null  float64
 7   median_income       20433 non-null  float64
 8   median_house_value  20433 non-null  float64
 9   ocean_proximity     20433 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.7+ MB


# 데이터 전처리

In [9]:
# 8이 결과이고 나머지가 독립변인이므로 
# X, y
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

## 수치형과 범주형으로 나눔
#num_features = X.drop('ocean_proximity', axis=1)
#cat_features = ['ocean_proximity']

## 수치형과 범주형으로 나눔(리스트로 만드세요!)
num_features = X.select_dtypes(include=["float64", "int64", "float32", "int32"]).columns.to_list()
cat_features = ['ocean_proximity']

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

In [11]:
X_train_num = X_train[num_features]
X_train_cat = X_train[cat_features]

X_test_num = X_test[num_features]
X_test_cat = X_test[cat_features]

In [12]:
# Standard는 학습하는 데이터의 독립변인 something을 넣어야 한다는것 유념!
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
X_train_num_scaled = scalar.fit_transform(X_train_num)
X_test_num_scaled = scalar.fit_transform(X_test_num)

In [13]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
X_train_cat_encoder = encoder.fit_transform(X_train_cat).toarray() # sklearn의 model에 인자로 넣을 거니 numpy array여야 한다
X_test_cat_encoder = encoder.fit_transform(X_test_cat).toarray()

In [14]:
X_train_num_scaled

array([[-1.107,  0.786, -1.162, ...,  1.131,  1.041,  0.438],
       [-0.025,  0.468,  0.349, ..., -0.709, -0.856, -0.242],
       [ 0.758, -0.712, -0.287, ...,  0.378,  0.692, -0.109],
       ...,
       [ 0.579, -0.763,  1.064, ..., -0.415, -0.359, -0.407],
       [-1.226,  0.903, -1.321, ...,  1.785,  1.48 ,  0.747],
       [-1.421,  0.978,  1.859, ...,  0.752,  0.395,  0.012]],
      shape=(16346, 8))

In [15]:
X_train_processed = np.hstack([X_train_num_scaled, X_train_cat_encoder])
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat_encoder])

# 모델링

In [16]:
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_model(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f" {name} ")
    print(f" -- r2  : {r2:.2f}")
    print(f" -- mse  : {mse:.2f}")
    print(f" -- rmse  : {rmse:.2f}")
    return {"model": name, "r2": r2, "rmse": rmse}

In [17]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_processed, y_train)
lr_ypred = lr.predict(X_test_processed)

result_lr = evaluate_model("Linear Regression", y_test, lr_ypred)
result_lr

 Linear Regression 
 -- r2  : 0.65
 -- mse  : 4808231332.03
 -- rmse  : 69341.41


{'model': 'Linear Regression',
 'r2': 0.648397238233239,
 'rmse': np.float64(69341.41137901986)}

In [18]:
# 범주형은 제외해야 합니다. 수치형만 가능해요!
from sklearn.preprocessing import PolynomialFeatures

#poly = PolynomialFeatures(degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_train_num_poly = poly.fit_transform(X_train_num_scaled) # include_bias=False를 하지 않고 앞에 붙는 1은 절편이다! 모델에서 절편 따로 관리하기 때문에 필요 없다!
X_test_num_poly = poly.fit_transform(X_test_num_scaled)

> interaction_only=False
|X1 |X2 |X[1,2]**2|
|==*|==*|==*|
|1  |2  |         |
|3  |4  |         |

> interaction_only=True
|X1 |X2 |X1**2|X2**2|X[1,2]**2|
|==*|==*|==*|==*|==*|
|1  |2  |1    |4    |         |
|3  |4  |9    |16   |         |

In [19]:
X_train_poly_processed = np.hstack([X_train_num_poly, X_train_cat_encoder])
X_test_poly_processed = np.hstack([X_test_num_poly, X_test_cat_encoder])

In [20]:
lr.fit(X_train_poly_processed, y_train)
lr_ypred = lr.predict(X_test_poly_processed)

result_lr = evaluate_model("Linear Regression (Poly)", y_test, lr_ypred)
result_lr

 Linear Regression (Poly) 
 -- r2  : 0.69
 -- mse  : 4240823852.48
 -- rmse  : 65121.61


{'model': 'Linear Regression (Poly)',
 'r2': 0.6898890099645132,
 'rmse': np.float64(65121.60818404155)}

In [21]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train_processed, y_train)
dt_ypred = dt.predict(X_test_processed)
result_dt = evaluate_model("Decision Tree Regressor", y_test, dt_ypred)
result_dt

 Decision Tree Regressor 
 -- r2  : 0.46
 -- mse  : 7359992187.03
 -- rmse  : 85790.40


{'model': 'Decision Tree Regressor',
 'r2': 0.4617992769401237,
 'rmse': np.float64(85790.39682287253)}

In [22]:
dt.feature_importances_

array([0.121, 0.12 , 0.116, 0.12 , 0.112, 0.121, 0.114, 0.152, 0.009,
       0.002, 0.   , 0.006, 0.008])

In [23]:
num_names = num_features
num_names

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income']

In [24]:
cat_names = list(encoder.get_feature_names_out(cat_features))
cat_names

['ocean_proximity_<1H OCEAN',
 'ocean_proximity_INLAND',
 'ocean_proximity_ISLAND',
 'ocean_proximity_NEAR BAY',
 'ocean_proximity_NEAR OCEAN']

In [25]:
all_names = num_names + cat_names
all_names

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'ocean_proximity_<1H OCEAN',
 'ocean_proximity_INLAND',
 'ocean_proximity_ISLAND',
 'ocean_proximity_NEAR BAY',
 'ocean_proximity_NEAR OCEAN']

In [26]:
importances = dt.feature_importances_
fi = pd.DataFrame({
    "features": all_names,
    "importances": importances
}).sort_values("importances", ascending=False)

In [27]:
fi

,features,importances
7,median_income,1.52e-01
0,longitude,1.21e-01
5,population,1.21e-01
3,total_rooms,1.20e-01
1,latitude,1.20e-01
2,housing_median_age,1.16e-01
6,households,1.14e-01
4,total_bedrooms,1.12e-01
8,ocean_proximity_<1H OCEAN,8.60e-03
12,ocean_proximity_NEAR OCEAN,8.38e-03


In [28]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(X_train_processed, y_train)
rf_ypred = rf.predict(X_test_processed)
result_rf = evaluate_model("RandomForest Regressor", y_test, rf_ypred)
result_rf

 RandomForest Regressor 
 -- r2  : 0.76
 -- mse  : 3233081575.50
 -- rmse  : 56860.19


{'model': 'RandomForest Regressor',
 'r2': 0.763580341197298,
 'rmse': np.float64(56860.193241870875)}

In [29]:
importances = rf.feature_importances_
fi = pd.DataFrame({
    "features": all_names,
    "importances": importances
}).sort_values("importances", ascending=False)

fi

,features,importances
7,median_income,4.85e-01
9,ocean_proximity_INLAND,1.43e-01
0,longitude,1.11e-01
1,latitude,1.04e-01
2,housing_median_age,5.08e-02
5,population,3.24e-02
3,total_rooms,2.35e-02
4,total_bedrooms,2.11e-02
6,households,1.88e-02
12,ocean_proximity_NEAR OCEAN,5.87e-03


### 데이터분석에서는 독립변인을 차원으로 본다 >

In [30]:
demo_df = pd.DataFrame({"숫자특성": [0,1,2,1],
                        "범주형": ["a", "b", "c", "a"]})

In [31]:
demo_df

,숫자특성,범주형
0,0,a
1,1,b
2,2,c
3,1,a


## 범주형
### get_dummies는 판다스로 하는 원핫인코딩
### 데이터분석용외에 pandas로 하지 않는다.
### 그 외에는 ski-learn을 사용한다.

In [32]:
pd.get_dummies(demo_df)

,숫자특성,범주형_a,범주형_b,범주형_c
0,0,True,False,False
1,1,False,True,False
2,2,False,False,True
3,1,True,False,False


### ski-learn으로
### 희소행렬로 받을지 numpy배열로 받을지